In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL
import math
import json
import time

root = Path.cwd()/'institutional-roi-analysis'
# os.chdir(root/"Seb_branch"/"institutional-roi-analysis"/"notebooks")
pd.set_option("display.max_columns",None)
display(root)

In [ ]:
def get_with_retries(url, params, tries=5, timeout=30):
    last = None
    for i in range(tries):
        r = requests.get(
            url,
            params=params,
            timeout=timeout,
            headers={"Accept": "application/json"},
        )
        if r.status_code < 500 and r.status_code != 429:
            return r
        if r.status_code == 429:
            time.sleep(5 * (i + 1))
            continue
        last = r
        time.sleep((2 ** i) + random.random())
    return last


def get_json_or_raise(response: requests.Response):
    try:
        response.raise_for_status()
    except requests.HTTPError as e:
        ct = response.headers.get("Content-Type", "")
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"HTTP {response.status_code} for {response.url}\n"
            f"Content-Type: {ct}\n"
            f"Body preview:\n{body_preview}"
        ) from e

    ct = response.headers.get("Content-Type", "")
    if "json" not in ct.lower():
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"Expected JSON but got Content-Type: {ct}\n"
            f"URL: {response.url}\n"
            f"Body preview:\n{body_preview}"
        )

    try:
        return response.json()
    except json.JSONDecodeError as e:
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"JSON decode failed for {response.url}\n"
            f"Body preview:\n{body_preview}"
        ) from e


def collect_school_level(per_page: int = 100, api_key: str = "") -> pd.DataFrame:
    fields = ",".join([
        "id",
        "school.name",
        "unit",
        
        "latest.academics.program_percentage.agriculture",
        "latest.academics.program_percentage.resources",
        "latest.academics.program_percentage.architecture",
        "latest.academics.program_percentage.ethnic_cultural_gender",
        "latest.academics.program_percentage.communication",
        "latest.academics.program_percentage.communications_technology",
        "latest.academics.program_percentage.computer",
        "latest.academics.program_percentage.personal_culinary",
        "latest.academics.program_percentage.education",
        "latest.academics.program_percentage.engineering",
        "latest.academics.program_percentage.engineering_technology",
        "latest.academics.program_percentage.language",
        "latest.academics.program_percentage.family_consumer_science",
        "latest.academics.program_percentage.legal",
        "latest.academics.program_percentage.english",
        "latest.academics.program_percentage.humanities",
        "latest.academics.program_percentage.library",
        "latest.academics.program_percentage.biological",
        "latest.academics.program_percentage.mathematics",
        "latest.academics.program_percentage.military",
        "latest.academics.program_percentage.multidiscipline",
        "latest.academics.program_percentage.parks_recreation_fitness",
        "latest.academics.program_percentage.philosophy_religious",
        "latest.academics.program_percentage.theology_religious_vocation",
        "latest.academics.program_percentage.physical_science",
        "latest.academics.program_percentage.science_technology",
        "latest.academics.program_percentage.psychology",
        "latest.academics.program_percentage.security_law_enforcement",
        "latest.academics.program_percentage.public_administration_social_service",
        "latest.academics.program_percentage.social_science",
        "latest.academics.program_percentage.construction",
        "latest.academics.program_percentage.mechanic_repair_technology",
        "latest.academics.program_percentage.precision_production",
        "latest.academics.program_percentage.transportation",
        "latest.academics.program_percentage.visual_performing",
        "latest.academics.program_percentage.health",
        "latest.academics.program_percentage.business_marketing",
        "latest.academics.program_percentage.history",
        "latest.school.instructional_expenditure_per_fte",
        "latest.school.faculty_salary",
        "latest.school.ft_faculty_rate",
        "latest.academics.program_reporter.programs_offered",
        "latest.student.demographics.student_faculty_ratio",
        "latest.school.endowment.begin",
        "latest.school.endowment.end",
        "latest.school.dolflag",

    ])

    params = {
        "api_key": SCORECARD_KEY,
        "fields": fields,
        "per_page": str(per_page),
        "page": "0",
    }

    response = get_with_retries(BASE_URL, params=params, timeout=30)
    data = get_json_or_raise(response)

    total = int(data["metadata"]["total"])
    per_page_actual = int(data["metadata"]["per_page"])
    total_pages = math.ceil(total / per_page_actual)

    rows = []

    for page in range(total_pages):
        params["page"] = str(page)
        response = get_with_retries(BASE_URL, params=params, timeout=30)
        data = get_json_or_raise(response)
        rows.extend(data.get("results", []))

    df = pd.json_normalize(rows)

    print(f"Total pages fetched: {total_pages}")
    print(f"Total schools ingested: {len(df)}")
    return df

In [ ]:
tdf=collect_school_level()

In [ ]:
tdf=clean(tdf)

In [ ]:
tdf=tdf.rename(columns={"id":"unit_id"})
save(tdf,file_name="national_inst_driver")